## 로그오즈비
## 각 동사별로 장르에 따라 었 출현여부가 달라지는지 로그오즈비를 구함.
동사별로, 장르에 따른 었 출현율 오즈비를 구하는 것.
"""
이 스크립트는 동사(unit) × 장르(context)별로
과거 시제 선어말어미 '-었-'의 출현 여부(outcome)를 기준으로
로그오즈비(log-odds ratio)를 계산한다.

[기본 아이디어]
- 분석은 항상 "unit별 조건부 비교"로 이루어진다.
- 각 Unit u에 대해, 각 Context g마다
  g vs (g를 제외한 나머지 Context들)에서의
  '-었-' 출현 odds를 비교한다.

[분할표 구성 (Unit u, Context g)]
    a = u & g & outcome=True   (예: '-었-' 출현)
    b = u & g & outcome=False
    c = u & other_contexts & outcome=True
    d = u & other_contexts & outcome=False

  log OR(u, g) = log((a*d)/(b*c))

[Outcome 처리]
- outcome은 반드시 이진(True/False)으로 처리된다.
- 범주형 outcome(예: '었/았/겠/∅')은
  positive_fn을 통해 one-vs-rest 방식으로 이진화해야 한다.

[집계 방식(count_mode)]
- count_mode="weight":
    · 각 셀의 값은 weight_col(ID)의 합으로 계산됨
    · 로그오즈비 계산과 unit/context 선택 기준이 모두 ID 합 기준
- count_mode="rows":
    · 각 셀의 값은 행 개수로 계산됨
    · 로그오즈비 계산과 unit/context 선택 기준이 모두 행 수 기준

※ 집계 방식과 unit/context 선택 기준은 항상 일치하도록 설계됨
  (weight로 계산하면서 행 수로 unit을 고르는 버그 방지)

[Unit / Context 선택]
- unit_values, context_values를 직접 주지 않으면
  다음 기준으로 자동 선택됨:
    · unit_top_n / context_top_n : 출현량 상위 N개
    · unit_min_count / context_min_count : 최소 출현량 기준
- 출현량의 정의는 count_mode에 따라 결정됨
  (rows → 행 수, weight → weight 합)

[추가 통계]
- 선택적으로 동사별 2×K (장르) 카이제곱 검정 수행
- Cramér’s V를 함께 계산하여 장르 효과의 크기를 요약

[이 스크립트가 제공하는 것]
- 각 (동사, 장르) 쌍에 대한 로그오즈비 값
- “이 동사가 특정 장르에서 '-었-'을 상대적으로 더/덜 쓰는가”에 대한
  정량적 비교 지표

[이 스크립트가 직접 제공하지 않는 것]
- 동사 하나에 대한 단일 ‘장르 민감성’ 점수
  (이는 log-odds 결과를 후처리하여 별도로 계산해야 함)
"""

# 
# chi2_df
    - unit_total = 총 빈도(=ID 합)
    - pos_total = -었- 결합 빈도(=ID 합)
    - neg_total = 비결합 빈도(=ID 합)
    - pos_rate = -었- 결합 비율 (pos_total / unit_total)
    - n_context_used = 실제로 관측된 장르 수 (빈도 0인 장르는 제외됨)
    - chi2, p_context_diff = “장르에 따라 결합 분포가 달라 보이냐”의 검정 요약
    - cramers_v = 효과크기(0~1)
    0에 가까우면: 장르 효과 거의 없음
    커질수록: 장르에 따른 결합 편차가 큼
    chi2랑 달리 unit 간 비교가 훨씬 안전함

In [1]:
# 1. Provide core functions for log-odds computation
# 
##로그오즈비
##각 동사별로 장르에 따라 었 출현여부가 달라지는지 로그오즈비를 구함.
import math
import pandas as pd
import numpy as np
from scipy import stats
from typing import Callable, Iterable, Optional, Dict, Any, Tuple, Union, Literal
import sys
from pathlib import Path
ROOT = Path.cwd().parents[0]   # 상위폴더를 ROOT로 넣음.
sys.path.append(str(ROOT))
from stats.filtering import apply_filters, FilterValue
# =========================================================
# Core stats
# =========================================================
def log_odds_2x2(
    a: float, b: float, c: float, d: float,
    alpha: float = 0.5,
    add_alpha_if_zero: bool = True
) -> Tuple[float, float, float, float, float]:
    """
    Returns (odds_ratio, log_or, se, z, p_value)
    If any cell is 0 and add_alpha_if_zero=True, adds alpha to ALL four cells.
    """
    if add_alpha_if_zero and (a == 0 or b == 0 or c == 0 or d == 0):
        a, b, c, d = a + alpha, b + alpha, c + alpha, d + alpha

    odds_ratio = (a * d) / (b * c)
    log_or = math.log(odds_ratio)
    se = math.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    z = log_or / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return odds_ratio, log_or, se, z, p


def default_binary_parser(x: Any) -> bool:
    """
    Generic robust binary parsing:
    - numeric: nonzero = True
    - strings: common truthy forms
    Customize this if your phenomenon has specific coding.
    """
    if pd.isna(x):
        return False
    if isinstance(x, (int, np.integer)):    #x가 int이거나 numpy 정수형(np.int64, np.int32 등)이면 
        return x != 0
    if isinstance(x, (float, np.floating)): #x가 float이거나 numpy 부동소수형(np.float64, np.float32 등)이면
        return x != 0.0
    s = str(x).strip().lower()
    return s in {"1", "true", "t", "y", "yes", "on"}

# =========================================================
# Helpers: Top-N selection by frequency
# =========================================================
def _topn_values(
    df: pd.DataFrame,
    col: str,
    *,
    mode: Literal["rows", "weight"] = "weight",
    weight_col: str = "ID",
    top_n: Optional[int] = None,
    min_count: Optional[float] = None,   # rows면 '최소 행수', weight면 '최소 가중합'
    dropna: bool = True
) -> list:
    """
    Pick values by frequency, consistent with aggregation mode.
    - mode="rows": frequency = number of rows
    - mode="weight": frequency = sum(weight_col)

    min_count means:
      - rows: minimum row count
      - weight: minimum weight sum
    """

    if dropna:
        d = df.dropna(subset=[col]).copy()
    else:
        d = df.copy()

    if mode == "rows":
        vc = d[col].value_counts(dropna=not dropna)

    elif mode == "weight":
        if weight_col not in d.columns:
            raise ValueError(f"weight_col '{weight_col}' not in dataframe.")
        vc = d.groupby(col, dropna=not dropna)[weight_col].sum().sort_values(ascending=False)

    else:
        raise ValueError("mode must be 'rows' or 'weight'.")

    if min_count is not None:
        vc = vc[vc >= float(min_count)]

    if top_n is not None:
        vc = vc.iloc[: int(top_n)]

    return vc.index.tolist()


#=========================================================
# Main engine
# =========================================================
def compute_logodds_by_unit_by_context(
    df: pd.DataFrame,
    *,
    unit_col: str,                                      # e.g., "V_No"
    context_col: str,                                   # e.g., "category"
    outcome_col: str,                                   # e.g., "EP_T"
    positive_fn: Callable[[Any], bool] = default_binary_parser,

    # --- Counting mode ---
    count_mode: Literal["auto", "rows", "weight"] = "auto",
    weight_col: str = "ID", # e.g., "ID" (only used if count_mode="weight")

    # --- Filters  ---
    filters: Optional[Dict[str, Any]] = None,

    # --- Unit/context selection  ---
    unit_top_n: Optional[int] = None,                    # e.g., 1000 (by frequency)
    unit_min_count: Optional[int] = None,                # alternatively threshold

    context_top_n: Optional[int] = None,                 # optional
    context_min_count: Optional[int] = None,             # optional

    # --- Log-odds parameters (same as before) ---
    alpha: float = 0.5,
    add_alpha_if_zero: bool = True,
    do_chi2: bool = True
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    """
    For each unit u and each context g:
      compares context g vs all other contexts within the same unit u,
      outcome=True vs outcome=False,

    Counting:
      - count_mode="weight": weighted counts = sum(weight_col)
      - count_mode="rows":  unweighted counts = row counts

    2x2 per (u,g):
      a = u & g & outcome(True)
      b = u & g & outcome(False)
      c = u & other_contexts & outcome(True)
      d = u & other_contexts & outcome(False)

    Returns:
      (result_df, chi2_df or None)
    """

    # 0. 모드 확인, Validate columns 지정한 열이 있는지 확인 ---
    if count_mode == "auto":
        count_mode = "weight" if weight_col else "rows"    
    
    needed = {unit_col, context_col, outcome_col}
    if count_mode == "weight":
        needed.add(weight_col)

    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    f = df.copy()

    # 1. Apply filters (reusable)필터 적용 ---
    
    if filters:
        f = apply_filters(f, filters=filters)
            

    # 2. Restrict to requested units/contexts 유닛과 맥락 제한---
    
    # --- Auto-detect unit/context values if not provided ---
    unit_values = _topn_values(
        f, unit_col,
        mode=count_mode,          # ✅ 집계 모드와 동일
        weight_col=weight_col,
        top_n=unit_top_n,
        min_count=unit_min_count,
        dropna=True
    )

    context_values = _topn_values(
        f, context_col,
        mode=count_mode,          # ✅ 집계 모드와 동일
        weight_col=weight_col,
        top_n=context_top_n,
        min_count=context_min_count,
        dropna=True
    )

    f = f[f[unit_col].isin(unit_values) & f[context_col].isin(context_values)].copy()

    # --- Compute binary outcome flag:outcome을 True/False로 만들기 ---
    bin_col = "_OUTCOME_ON"
    f[bin_col] = f[outcome_col].apply(positive_fn)

    # --- 3D weighted pivot: (unit, context, outcome) -> sum(weight) ---
    # (동사, 장르, outcome True/False) 조합별로 weight_col을 합계(sum) 내고 마지막 차원(outcome)을 열로 펼침(unstack)
    if count_mode == "weight":
        pivot = (
            f.groupby([unit_col, context_col, bin_col], dropna=False)[weight_col]
            .sum()
            .unstack(bin_col, fill_value=0)
        )
    elif count_mode == "rows":
        pivot = (
            f.groupby([unit_col, context_col, bin_col], dropna=False)
            .size()
            .unstack(bin_col, fill_value=0)
        )
    else:
        raise ValueError("count_mode must be 'rows' or 'weight'.")

    # Ensure both outcome states exist as columns: True/False 컬럼이 없을 때를 대비
    if True not in pivot.columns:
        pivot[True] = 0
    if False not in pivot.columns:
        pivot[False] = 0

    # Fill all unit×context combos: ensure all (u,g) exist-없는 unit×context 조합을 0으로 채워서 만들기
    idx = pd.MultiIndex.from_product([unit_values, context_values], names=[unit_col, context_col]) #튜플 형태로 된 (unit, context) 조합의 모든 가능한 곱집합을 만듦.
    pivot = pivot.reindex(idx, fill_value=0) #튜플 형태의 (unit, context) 조합의 모든 가능한 곱집합을 인덱스로 사용하여 피벗 테이블을 재색인. 존재하지 않는 조합은 0으로 채움.

    # --- Compute log-odds per (unit, context) 로그오즈비 계산 ---
    rows = []
    for u in unit_values:
        sub = pivot.loc[u].reindex(context_values)  # index: context, columns: {False, True}   
        u_pos_total = float(sub[True].sum())
        u_neg_total = float(sub[False].sum())

        for g in context_values:
            a = float(sub.loc[g, True])
            b = float(sub.loc[g, False])
            c = float(u_pos_total - a)
            d = float(u_neg_total - b)

            odds_ratio, log_or, se, z, p = log_odds_2x2(
                a, b, c, d,
                alpha=alpha,
                add_alpha_if_zero=add_alpha_if_zero
            )

            rows.append({
                unit_col: u,
                context_col: g,
                "a_pos_in_context": a,
                "b_neg_in_context": b,
                "c_pos_other_contexts": c,
                "d_neg_other_contexts": d,
                "odds_ratio": odds_ratio,
                "log_odds": log_or,
                "SE": se,
                "z": z,
                "p_value": p,
                "unit_pos_total": u_pos_total,
                "unit_neg_total": u_neg_total,
                "unit_total": u_pos_total + u_neg_total,
                "alpha": alpha if (add_alpha_if_zero and (a == 0 or b == 0 or c == 0 or d == 0)) else 0.0,
            })

    result_df = pd.DataFrame(rows)

    # --- Optional: per-unit chi-square 2xK (pos/neg × contexts) ---
    # --- Optional: per-unit chi-square 2xK (pos/neg × contexts) + Cramér’s V ---
    chi2_df = None
    if do_chi2:
        tests = []
        for u in unit_values:
            sub = pivot.loc[u].reindex(context_values)

            pos = sub[True].astype(float).values
            neg = sub[False].astype(float).values
            table = np.vstack([pos, neg])  # 2 x K

            # 1) drop empty genres (column sum == 0)
            col_sum = table.sum(axis=0)
            valid_cols = col_sum > 0
            table_valid = table[:, valid_cols]

            n_context_used = int(table_valid.shape[1])
            pos_total = float(table_valid[0].sum())
            neg_total = float(table_valid[1].sum())
            unit_total = pos_total + neg_total

            # 2) chi-square preconditions:
            # - at least 2 contexts
            # - non-empty table
            # - BOTH rows must have >0 total (otherwise expected table will contain zeros)
            if n_context_used < 2 or unit_total == 0 or pos_total == 0 or neg_total == 0:
                tests.append({
                    unit_col: u,
                    "n_context_used": n_context_used,
                    "unit_total": unit_total,
                    "pos_total": pos_total,
                    "neg_total": neg_total,
                    "pos_rate": (pos_total / unit_total) if unit_total > 0 else np.nan,
                    "chi2": np.nan,
                    "df": np.nan,
                    "p_context_diff": np.nan,
                    "cramers_v": np.nan,
                    "chi2_note": (
                        "skip: n_context_used<2 or unit_total==0 or pos_total==0 or neg_total==0"
                    ),
                })
                continue

            # 3) run chi-square safely
            try:
                chi2, p_ctx, dfree, _ = stats.chi2_contingency(table_valid)
                cramers_v = math.sqrt(float(chi2) / unit_total) if unit_total > 0 else np.nan

                tests.append({
                    unit_col: u,
                    "n_context_used": n_context_used,
                    "unit_total": unit_total,
                    "pos_total": pos_total,
                    "neg_total": neg_total,
                    "pos_rate": (pos_total / unit_total) if unit_total > 0 else np.nan,
                    "chi2": float(chi2),
                    "df": int(dfree),
                    "p_context_diff": float(p_ctx),
                    "cramers_v": float(cramers_v),
                    "chi2_note": "",
                })

            except ValueError as e:
                # catches expected==0 etc.
                tests.append({
                    unit_col: u,
                    "n_context_used": n_context_used,
                    "unit_total": unit_total,
                    "pos_total": pos_total,
                    "neg_total": neg_total,
                    "pos_rate": (pos_total / unit_total) if unit_total > 0 else np.nan,
                    "chi2": np.nan,
                    "df": np.nan,
                    "p_context_diff": np.nan,
                    "cramers_v": np.nan,
                    "chi2_note": f"skip: {str(e)}",
                })

        chi2_df = (
            pd.DataFrame(tests)
            .sort_values(["p_context_diff", "cramers_v", unit_col], na_position="last")
        )


    return result_df, chi2_df


In [ ]:
# =========================================================
# 2. Practice: your current use-case (-었- 결합, EP_T)
# =========================================================
CSV_PATH = r"..\csv\통계용\V_VX_Category_2026-01-08_14-22.csv"
#,V_form,V_label,category,file_id,EP_T,EN_form,EN_label,vx0_No,vx0_order,vx_len,ID
UNIT_COL ="V_form"
CONTEXT_COL="V_label"
OUTCOME_COL="EP_T"
WEIGHT_COL="ID"

UNIT_MIN_COUNT = 100      #동사 최소 출현량
CONTEXT_MIN_COUNT = 0  #장르 최소 출현량

filters = {
    "EN_label": "EF",
    "EN_No": lambda s: (s >= 1101) & (s < 1102), #다, 는다 한정
    "copus": 2,
    "category": ["상상-일반", "보도·해설"],

    #"V_No" : [3161, 3077, 3057, 3040, 3116, 3132, 3021, 3214, 3159, 3027, 3072, 3033,
    #          3014, 3008, 3079, 3011, 2056, 3012, 3016, 3024, 3047, 3001, 3020, 3022,
    #          3006, 3004, 3026, 3003, 3002, 2002, 2004, 3025, 3007, 2008, 2013, 2003,
    #          2005, 2014, 2001, 1001, 3017, 2006, 2015],
    #"category": ["보도·해설", "사설", "상상-아동", "상상-일반", "인문사회", "자연", "체험기술", "총류", "칼럼"]
}

df = pd.read_csv(CSV_PATH)

# EP_T is “-었- attached?” -> True/False
def past_positive(x):
    # if your EP_T is strictly 0/1, you can simplify to: return int(x) == 1
    return default_binary_parser(x)

res, chi2 = compute_logodds_by_unit_by_context(
    df,
    unit_col=UNIT_COL,
    context_col=CONTEXT_COL,
    outcome_col=OUTCOME_COL,
    weight_col=WEIGHT_COL,
    count_mode="weight",    
    positive_fn=past_positive,
    filters=filters,

    unit_min_count=UNIT_MIN_COUNT,   #동사 최소 출현량
    context_min_count=CONTEXT_MIN_COUNT, #장르 최소 출현량

    alpha=0.5,
    add_alpha_if_zero=True,
    do_chi2=True
)
print("\n[Preview] chi-square top 10:")
print(chi2.head(10).to_string(index=False))


[Preview] chi-square top 10:
 V_No  n_context_used  unit_total  pos_total  neg_total  pos_rate         chi2  df  p_context_diff  cramers_v chi2_note
  102               2     10130.0     5533.0     4597.0  0.546199  1838.306220 1.0    0.000000e+00   0.425995          
  103               2      8973.0     6524.0     2449.0  0.727070  1921.261599 1.0    0.000000e+00   0.462727          
 2004               2     10005.0     6014.0     3991.0  0.601099  2758.238883 1.0    0.000000e+00   0.525058          
 1001               2     60484.0    24582.0    35902.0  0.406422 16785.637932 1.0    0.000000e+00   0.526804          
 2001               2      9682.0     3315.0     6367.0  0.342388  3620.088218 1.0    0.000000e+00   0.611473          
  101               2     32656.0    14628.0    18028.0  0.447942 22270.207572 1.0    0.000000e+00   0.825811          
 4999               2     28719.0    22395.0     6324.0  0.779797  1391.883744 1.0   1.219305e-304   0.220149          
 3022     

In [42]:
# =========================================================
# 4. save to file
# =========================================================
from pathlib import Path
from datetime import datetime

#---- file name settings ----  
SAVE_DIR = r"csv\결과값"   

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
out_file = f'{OUTCOME_COL}_by_{UNIT_COL}_{CONTEXT_COL}_{WEIGHT_COL}.csv'

print(f"***저장합니다.:    {datetime.now()}***")

res.to_csv(SAVE_DIR + f"\{timestamp}_logodds_{out_file}", index=False, encoding="utf-8-sig")
if chi2 is not None:
    chi2.to_csv(SAVE_DIR + f"\{timestamp}_chi2_{out_file}", index=False, encoding="utf-8-sig")

    print(f"[OK] saved logodds_{out_file}")
    print(f"[OK] saved chi2_{out_file}")


***저장합니다.:    2025-12-25 09:00:37.927892***
[OK] saved logodds_EP_T_by_V_No_category_ID.csv
[OK] saved chi2_EP_T_by_V_No_category_ID.csv


<string>:15: SyntaxWarning: invalid escape sequence '\{'
<string>:17: SyntaxWarning: invalid escape sequence '\{'
<>:15: SyntaxWarning: invalid escape sequence '\{'
<>:17: SyntaxWarning: invalid escape sequence '\{'
<string>:15: SyntaxWarning: invalid escape sequence '\{'
<string>:17: SyntaxWarning: invalid escape sequence '\{'
<>:15: SyntaxWarning: invalid escape sequence '\{'
<>:17: SyntaxWarning: invalid escape sequence '\{'
C:\Users\yu2hy\AppData\Local\Temp\ipykernel_14364\2737182001.py:15: SyntaxWarning: invalid escape sequence '\{'
  res.to_csv(SAVE_DIR + f"\{timestamp}_logodds_{out_file}", index=False, encoding="utf-8-sig")
C:\Users\yu2hy\AppData\Local\Temp\ipykernel_14364\2737182001.py:17: SyntaxWarning: invalid escape sequence '\{'
  chi2.to_csv(SAVE_DIR + f"\{timestamp}_chi2_{out_file}", index=False, encoding="utf-8-sig")
